In [5]:
conda install -y -c conda-forge geopandas pyogrio shapely rtree pyproj fiona pyarrow

Retrieving notices: done
Channels:
 - conda-forge
Platform: osx-arm64
Solving environment: done


==> WARNING: A newer version of conda exists. <==
    current version: 24.11.3
    latest version: 25.9.1

Please update conda by running

    $ conda update -n base -c conda-forge conda



## Package Plan ##

  environment location: /Users/laurenvo/Documents/github/Mapping-Youth-Opportunity-Deserts/.conda

  added / updated specs:
    - fiona
    - geopandas
    - pyarrow
    - pyogrio
    - pyproj
    - rtree
    - shapely


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    aws-c-auth-0.9.1           |       h8818502_7         104 KB  conda-forge
    aws-c-cal-0.9.10           |       hca30140_1          44 KB  conda-forge
    aws-c-common-0.12.5        |       hc919400_1         218 KB  conda-forge
    aws-c-compression-0.3.1    |       h61d5560_8          21 KB  conda-forge
    aws-c-event

In [ ]:
# Imports & robust repo-root detection
from pathlib import Path
import zipfile
import pandas as pd
import numpy as np
import geopandas as gpd

def find_repo_root(start: Path | None = None) -> Path:
    p = (start or Path.cwd()).resolve()
    markers = ("data", ".git", "pyproject.toml", "Makefile")
    while p.parent != p and not any((p / m).exists() for m in markers):
        p = p.parent
    return p

REPO = find_repo_root()
DATA = REPO / "data"
RAW = DATA / "raw"
EXT = DATA / "external"
INTERIM = DATA / "interim"; INTERIM.mkdir(parents=True, exist_ok=True)
PROCESSED = DATA / "processed"; PROCESSED.mkdir(parents=True, exist_ok=True)

# Services 
SERV_RAW = RAW / "services"
YMCA_F   = SERV_RAW / "ymca_sd.csv"
LIBS_F   = SERV_RAW / "sd_public_libraries.csv"
LIBS2_F  = SERV_RAW / "Library.csv"                   
REC_F    = SERV_RAW / "rec_centers_datasd.csv"
REC2_F   = SERV_RAW / "Recreation_Centers_SD.csv"  
PARKS_F  = SERV_RAW / "Parks_SD.csv"
GTFS_STOPS_F  = SERV_RAW / "Transit_Stops_GTFS_20251028.csv"
GTFS_ROUTES_F = SERV_RAW / "Transit_Routes_GTFS.csv"

# TIGER 2023 shapefiles (California = 06)
TIGER_DIR = EXT / "tiger_tracts_2023"
TRACT_ZIP = TIGER_DIR / "tl_2023_06_tract.zip"    # tracts for CA
PLACE_ZIP = TIGER_DIR / "tl_2023_06_place.zip"    # cities/places
COUNTY_ZIP= TIGER_DIR / "tl_2023_us_county.zip"   # US counties

# ACS baseline 
ACS_CLEAN = PROCESSED / "acs_baselines_tract_2023_clean.csv"

print("Repo root:", REPO)
for p in [YMCA_F, LIBS_F, LIBS2_F, REC_F, REC2_F, PARKS_F, GTFS_STOPS_F, GTFS_ROUTES_F, TRACT_ZIP, ACS_CLEAN]:
    print(f"{p.relative_to(REPO)!s:<80} exists={p.exists()}")


False False False
Tracts file: None


In [ ]:
# Read tracts from the zipped shapefile directly (GeoPandas can read .zip via fiona)
tracts_ca = gpd.read_file(f"zip://{TRACT_ZIP}")
print(tracts_ca.crs)
tracts_ca = tracts_ca.to_crs(4326)

# Keep San Diego County only (STATEFP='06', COUNTYFP='073')
tracts_sd = tracts_ca.query("(STATEFP == '06') and (COUNTYFP == '073')").copy()

# Keep a compact set of ID columns for later joins
tracts_sd = tracts_sd[["STATEFP","COUNTYFP","TRACTCE","GEOID","NAME","NAMELSAD","ALAND","AWATER","geometry"]]
tracts_sd = tracts_sd.sort_values("GEOID").reset_index(drop=True)

print("San Diego tracts:", len(tracts_sd))
tracts_sd.head(2)


EPSG:4269
San Diego tracts: 737


,STATEFP,COUNTYFP,TRACTCE,GEOID,NAME,NAMELSAD,ALAND,AWATER,geometry
0,06,073,000100,06073000100,1,Census Tract 1,1536251,0,"POLYGON ((-117.1949 32.75278, -117.19471 32.75..."
1,06,073,000201,06073000201,2.01,Census Tract 2.01,864211,0,"POLYGON ((-117.17887 32.75765, -117.17797 32.7..."


In [ ]:
# 2) Utilities for consistent point ingestion
def best_col(cols, candidates):
    for c in candidates:
        if c in cols:
            return c
    return None

def read_points_csv(path: Path, source_name: str) -> gpd.GeoDataFrame:
    df = pd.read_csv(path, dtype=str, low_memory=False)
    # Infer lat/lon
    lat_col = best_col(df.columns, ["lat","latitude","stop_lat","y","Lat","Latitude"])
    lon_col = best_col(df.columns, ["lng","lon","longitude","stop_lon","x","Long","Longitude"])
    if lat_col is None or lon_col is None:
        raise ValueError(f"{source_name}: could not find latitude/longitude columns in {list(df.columns)[:12]}...")

    # Coerce to numeric; drop invalid
    df["_lat"] = pd.to_numeric(df[lat_col], errors="coerce")
    df["_lon"] = pd.to_numeric(df[lon_col], errors="coerce")
    df = df.dropna(subset=["_lat","_lon"]).copy()

    gdf = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df["_lon"], df["_lat"], crs="EPSG:4326"))
    gdf["source"] = source_name
    return gdf

def read_optional_points(path: Path, source_name: str) -> gpd.GeoDataFrame | None:
    if path.exists():
        try:
            return read_points_csv(path, source_name)
        except Exception as e:
            print(f"[WARN] {source_name}: {e}")
            return None
    else:
        return None


In [5]:
def pick_latlon(cols):
    cols_l = {c.lower(): c for c in cols}
    pairs = [
        ("lat", "lon"),
        ("lat", "lng"),
        ("latitude", "longitude"),
        ("y", "x"),
        ("point_y", "point_x"),
        ("stop_lat", "stop_lon"),
    ]
    for a, b in pairs:
        if a in cols_l and b in cols_l:
            return cols_l[a], cols_l[b]
    return None, None

def read_optional_points(csv_path: Path, label: str):
    if not csv_path.exists():
        print(f"[WARN] {label}: file missing -> {csv_path}")
        return None
    try:
        df = pd.read_csv(csv_path)
    except Exception as e:
        print(f"[WARN] {label}: could not read CSV ({e})")
        return None

    lat_col, lon_col = pick_latlon(df.columns)
    if lat_col is None or lon_col is None:
        print(f"[WARN] {label}: could not find latitude/longitude columns in {list(df.columns)}...")
        return None

    gdf = gpd.GeoDataFrame(
        df,
        geometry=gpd.points_from_xy(df[lon_col], df[lat_col]),
        crs="EPSG:4326",
    )

    # keep a small set of context columns when present
    keep_candidates = ["name", "org", "address", "city", "state", "zip", "website", "source"]
    keep = [c for c in keep_candidates if c in gdf.columns]
    return gdf[keep + ["geometry"]].copy()


In [ ]:
gdfs = []

# Libraries
libs = read_optional_points(LIBS2_F, "Libraries_SD_alt")
if libs is None:
    libs = read_optional_points(LIBS_F, "Libraries_SD")
if libs is not None:
    libs["category"] = "library"
    gdfs.append(libs)

# Recreation centers: try both in sequence
rec = read_optional_points(REC2_F, "Rec_Centers_alt")
if rec is None:
    rec = read_optional_points(REC_F, "Rec_Centers")
if rec is not None:
    rec["category"] = "rec_center"
    gdfs.append(rec)

# YMCA
ymca = read_optional_points(YMCA_F, "YMCA")
if ymca is not None:
    ymca["category"] = "ymca"
    gdfs.append(ymca)

# Parks 
parks = read_optional_points(PARKS_F, "Parks_SD")
if parks is not None:
    parks["category"] = "park"
    gdfs.append(parks)

# GTFS stops
gtfs = read_optional_points(GTFS_STOPS_F, "GTFS_Stops")
if gtfs is not None:
    gtfs["category"] = "transit_stop"
    gdfs.append(gtfs)

print("Layers loaded:", [g.shape for g in gdfs])
services_all = pd.concat(gdfs, ignore_index=True) if gdfs else gpd.GeoDataFrame(columns=["category", "geometry"], crs="EPSG:4326")
print("Total service points:", len(services_all))


[WARN] Libraries_SD_alt: could not find latitude/longitude columns in ['OID_', 'WEBSITE', 'ADDRESS', 'DISTRICT', 'NAME', 'TYPE', 'PHONE', 'ZIP', 'CITY', 'DATA_SRC', 'UPDATE_DATE']...
[WARN] Libraries_SD: could not find latitude/longitude columns in ['org', 'name', 'address', 'city', 'state', 'zip', 'phone', 'website', 'source']...
[WARN] YMCA: could not find latitude/longitude columns in ['org', 'name', 'address', 'city', 'state', 'zip', 'phone', 'fax', 'website', 'source']...
[WARN] Parks_SD: could not find latitude/longitude columns in ['objectid', 'common_name', 'desig_use', 'acres', 'address_lo', 'community', 'use_restriction', 'dedicated_park', 'full_name', 'undeveloped_ac', 'useable_ac', 'gross_ac', 'maint_entity', 'recycled_water', 'play_area_cnt', 'cd', 'dms', 'x_coord', 'y_coord', 'year_acquired', 'read_ac', 'notes', 'year_constructed', 'year_assessed', 'pci', 'service_district', 'playground', 'tot_lot', 'baseball_50_6', 'baseball_90', 'softball', 'multi_purpose', 'f_2_baselet

In [8]:
# Ensure same CRS
services_all = services_all.to_crs(4326)
tracts_sd = tracts_sd.to_crs(4326)

# Keep only points inside San Diego County tracts
points_in_sd = gpd.sjoin(services_all, tracts_sd[["GEOID","geometry"]], how="inner", predicate="within")
print("Points matched to SD tracts:", len(points_in_sd))

# Save a GeoJSON of the matched points (handy for quick map checks)
points_geojson = INTERIM / "services_points_in_sd_2023.geojson"
points_in_sd.to_file(points_geojson, driver="GeoJSON")
print("Wrote:", points_geojson)


Points matched to SD tracts: 6230
Wrote: /Users/laurenvo/Documents/github/Mapping-Youth-Opportunity-Deserts/data/interim/services_points_in_sd_2023.geojson


In [ ]:
# Count by tract & category
counts = (
    points_in_sd
      .groupby(["GEOID","category"], as_index=False)
      .size()
      .rename(columns={"size":"count"})
)

# Pivot to wide columns (ymca_ct, library_ct, rec_center_ct, park_ct, gtfs_stop_ct)
wide = counts.pivot(index="GEOID", columns="category", values="count").fillna(0).astype(int)
wide.columns = [f"{c}_ct" for c in wide.columns]
wide = wide.reset_index()

# Add a total services column excluding GTFS unless desired
service_cols = [c for c in wide.columns if c.endswith("_ct") and c != "gtfs_stop_ct"]
wide["services_ct"] = wide[service_cols].sum(axis=1)

wide.head()

,GEOID,transit_stop_ct,services_ct
0,06073000100,7,7
1,06073000201,7,7
2,06073000202,11,11
3,06073000301,7,7
4,06073000302,10,10


In [ ]:
# Load canonical tract-level baseline
acs = pd.read_csv(ACS_CLEAN, dtype={"GEOID":"string"})

# Merge
tract_services = tracts_sd[["GEOID"]].merge(wide, on="GEOID", how="left").fillna(0)
tract_services = tract_services.merge(
    acs[["GEOID","NAME","pop_total","youth_5_17","youth_10_19","youth_5_17_per_1k","youth_10_19_per_1k"]],
    on="GEOID",
    how="left"
)

# Per-1k youth (guard zeros)
def per_1k(n, denom):
    n = n.astype("float64")
    d = denom.astype("float64")
    out = np.where((d > 0) & np.isfinite(d), 1000.0 * n / d, np.nan)
    return out

# Compute per-1k for key categories against 5–17 and 10–19
for base_col in ["ymca_ct","library_ct","rec_center_ct","park_ct","services_ct"]:
    if base_col in tract_services.columns:
        tract_services[f"{base_col}_per_1k_5_17"]  = per_1k(tract_services[base_col], tract_services["youth_5_17"])
        tract_services[f"{base_col}_per_1k_10_19"] = per_1k(tract_services[base_col], tract_services["youth_10_19"])

# Include GTFS stop density per 10k population as an access proxy
if "gtfs_stop_ct" in tract_services.columns:
    tract_services["gtfs_stop_ct_per_10k_pop"] = per_1k(tract_services["gtfs_stop_ct"], tract_services["pop_total"]) * 10

# Order columns for readability
front = ["GEOID","NAME","pop_total","youth_5_17","youth_10_19","youth_5_17_per_1k","youth_10_19_per_1k"]
count_cols = [c for c in tract_services.columns if c.endswith("_ct")]
rate_cols  = [c for c in tract_services.columns if c.endswith("_per_1k_5_17") or c.endswith("_per_1k_10_19") or c.endswith("_per_10k_pop")]
other_cols = [c for c in tract_services.columns if c not in front+count_cols+rate_cols]
tract_services = tract_services[front + sorted(count_cols) + sorted(rate_cols) + sorted(other_cols)]

print("Rows:", len(tract_services))
tract_services.head(3)


Rows: 737


,GEOID,NAME,pop_total,youth_5_17,youth_10_19,youth_5_17_per_1k,youth_10_19_per_1k,services_ct,transit_stop_ct,services_ct_per_1k_10_19,services_ct_per_1k_5_17
0,06073000100,Census Tract 1; San Diego County; California,2746.0,360.0,310.0,131.099782,112.891479,7.0,7.0,22.580645,19.444444
1,06073000201,Census Tract 2.01; San Diego County; California,2376.0,355.0,279.0,149.410774,117.424242,7.0,7.0,25.089606,19.718310
2,06073000202,Census Tract 2.02; San Diego County; California,4019.0,344.0,247.0,85.593431,61.458074,11.0,11.0,44.534413,31.976744


In [ ]:
# CSV for modeling / dashboard plumbing
out_csv = PROCESSED / "services_by_tract_2023.csv"
tract_services.to_csv(out_csv, index=False)
print("Wrote:", out_csv)

# Write a GeoPackage with tract geometries + counts for quick mapping
tracts_counts = tracts_sd.merge(tract_services, on="GEOID", how="left")
out_gpkg = PROCESSED / "services_by_tract_2023.gpkg"
tracts_counts.to_file(out_gpkg, layer="tract_services", driver="GPKG")
print("Wrote:", out_gpkg)

# Quick summaries
display(tract_services.describe(include="number").T.head(12))
for col in ["ymca_ct","library_ct","rec_center_ct","park_ct","services_ct"]:
    if col in tract_services.columns:
        print(f"{col}: nonzero tracts = {(tract_services[col]>0).sum()} / {len(tract_services)}")


Wrote: /Users/laurenvo/Documents/github/Mapping-Youth-Opportunity-Deserts/data/processed/services_by_tract_2023.csv
Wrote: /Users/laurenvo/Documents/github/Mapping-Youth-Opportunity-Deserts/data/processed/services_by_tract_2023.gpkg


,count,mean,std,min,25%,50%,75%,max
pop_total,737.0,4454.249661,2107.362836,0.0,3313.000000,4257.000000,5436.000000,40481.000000
youth_5_17,737.0,687.552239,437.399949,0.0,422.000000,633.000000,897.000000,4064.000000
youth_10_19,737.0,553.706920,426.976540,0.0,316.000000,494.000000,703.000000,5984.000000
youth_5_17_per_1k,735.0,148.527286,60.066114,0.0,116.088617,155.036095,187.048111,338.547747
youth_10_19_per_1k,735.0,119.281729,59.520336,0.0,85.907905,117.724519,146.556075,688.869153
services_ct,737.0,8.453189,9.007593,0.0,3.000000,7.000000,11.000000,90.000000
transit_stop_ct,737.0,8.453189,9.007593,0.0,3.000000,7.000000,11.000000,90.000000
services_ct_per_1k_10_19,730.0,36.670004,129.648161,0.0,5.649718,13.640487,26.306185,2000.000000
services_ct_per_1k_5_17,728.0,27.798762,86.122625,0.0,4.379034,10.605329,21.178180,1444.444444


services_ct: nonzero tracts = 650 / 737
